## >>> VERSION: 2026-07-29 · stage5-detection-v2 (adds class histogram) <<<
**Stage 5 — downstream object detection on the six image versions (INSTRUMENTED).**

Runs a COCO-pretrained **YOLOv8x** detector on each of the six versions of all 500 images:
clean ground truth, degraded input, DiT, RCDNet, RCDNet→DiT, DiT→RCDNet. Detections on the
**clean** image are the pseudo-reference (Raindrop Clarity has no object labels). For every
other version we match detections to the clean reference (same class, IoU ≥ 0.5) and record
number of objects, mean confidence, false positives, false negatives, and precision / recall /
F1 (agreement) against clean. Each version is resized to the clean image's size first, so object
scales are comparable.

**Attach two inputs:** `500-drop-clear-zip` (provides `Clear/` = clean and `Drop/` = degraded) and
`stage5-restorations` (provides `dit/ rcdnet/ rcdnet_then_dit/ dit_then_rcdnet/`). **GPU T4, Internet on.**

**Outputs:** `metrics.zip` with `detection_per_image.csv`, `detection_summary.csv`, and
`stage4_detection_target.csv` (per-image best restoration version by detection F1, which feeds the
Stage 4 selector re-run).

In [ ]:
!pip install ultralytics -q

In [ ]:
# locate the six version roots under /kaggle/input
import os, glob
def find_dir(name):
    for d, subs, _ in os.walk('/kaggle/input'):
        if os.path.basename(d) == name and glob.glob(os.path.join(d, '**', '*.png'), recursive=True):
            return d
    return None
ROOT = {}
ROOT['clean']    = find_dir('Clear')
ROOT['degraded'] = find_dir('Drop')
for v in ['dit', 'rcdnet', 'rcdnet_then_dit', 'dit_then_rcdnet']:
    ROOT[v] = find_dir(v)
for v, p in ROOT.items():
    assert p, f'could not find version folder: {v}'
    print(f'{v:16s} {p}')
# keys from the clean folder (id/frame.png), then confirm every version has all of them
KEYS = sorted(os.path.relpath(p, ROOT['clean']).replace(os.sep, '/')
              for p in glob.glob(os.path.join(ROOT['clean'], '**', '*.png'), recursive=True))
print('keys:', len(KEYS))
for v in ROOT:
    miss = [k for k in KEYS if not os.path.exists(os.path.join(ROOT[v], k))]
    assert not miss, f'{v} missing {len(miss)} (e.g. {miss[:3]})'
print('all six versions have all', len(KEYS), 'images')
os.makedirs('/kaggle/working/metrics', exist_ok=True)

In [ ]:
from ultralytics import YOLO
import torch
print('CUDA:', torch.cuda.is_available())
model = YOLO('yolov8x.pt')          # COCO-pretrained, auto-downloads
DEV = 0 if torch.cuda.is_available() else 'cpu'
CONF = 0.25          # detection confidence threshold
IOU_MATCH = 0.5      # IoU for matching a detection to the clean reference
VERS = ['clean', 'degraded', 'dit', 'rcdnet', 'rcdnet_then_dit', 'dit_then_rcdnet']
ROUTABLE = ['degraded', 'dit', 'rcdnet', 'rcdnet_then_dit', 'dit_then_rcdnet']  # clean is the reference, not a choice
print('smoke test on first clean image...')
_r = model.predict(os.path.join(ROOT['clean'], KEYS[0]), conf=CONF, verbose=False, device=DEV)[0]
print('  objects on', KEYS[0], '=', len(_r.boxes))
print('PLUMBING OK')

In [ ]:
from PIL import Image
import numpy as np, csv, math, time

def iou(a, b):
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1]); ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1); inter = iw * ih
    ua = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - inter
    return inter / ua if ua > 0 else 0.0

def extract(res):
    out = []
    for bx in res.boxes:
        out.append((int(bx.cls[0]), float(bx.conf[0]), [float(x) for x in bx.xyxy[0].tolist()]))
    return out

def matched_tp(ref, det):
    used = [False] * len(ref); tp = 0
    for c, cf, box in sorted(det, key=lambda x: -x[1]):
        best, bi = IOU_MATCH, -1
        for i, (rc, rbox) in enumerate(ref):
            if used[i] or rc != c: continue
            v = iou(box, rbox)
            if v >= best: best, bi = v, i
        if bi >= 0: used[bi] = True; tp += 1
    return tp

rows = []            # per (image, version)
target = []          # per image: best routable version by F1
n_noref = 0
from collections import Counter
cls_count = {v: Counter() for v in VERS}   # class histogram per version (for 'top false classes')
t0 = time.time()
for n, k in enumerate(KEYS):
    cimg = Image.open(os.path.join(ROOT['clean'], k)).convert('RGB'); W, H = cimg.size
    imgs = []
    for v in VERS:
        im = Image.open(os.path.join(ROOT[v], k)).convert('RGB')
        if im.size != (W, H): im = im.resize((W, H))
        imgs.append(im)
    res = model.predict(imgs, conf=CONF, iou=0.5, verbose=False, device=DEV)
    dets = {v: extract(res[i]) for i, v in enumerate(VERS)}
    for v in VERS:
        for (c, cf, box) in dets[v]: cls_count[v][model.names[int(c)]] += 1
    ref = [(c, box) for (c, cf, box) in dets['clean']]; nref = len(ref)
    if nref == 0: n_noref += 1
    f1s = {}
    for v in VERS:
        d = dets[v]; nobj = len(d); mc = float(np.mean([x[1] for x in d])) if d else 0.0
        tp = nref if v == 'clean' else matched_tp(ref, d)
        fp = nobj - tp; fn = nref - tp
        prec = tp / (tp + fp) if (tp + fp) > 0 else (1.0 if nref == 0 else 0.0)
        rec = (tp / nref) if nref > 0 else float('nan')
        f1 = (2 * prec * rec / (prec + rec)) if (nref > 0 and prec + rec > 0) else float('nan')
        f1s[v] = f1
        rows.append([k, v, nobj, round(mc, 4), nref, tp, fp, fn,
                     round(prec, 4), ('' if nref == 0 else round(rec, 4)),
                     ('' if nref == 0 else round(f1, 4))])
    # stage-4 detection target: best routable version by F1 (only when a reference exists)
    if nref > 0:
        best = max(ROUTABLE, key=lambda v: (f1s[v] if not math.isnan(f1s[v]) else -1))
        target.append([k, nref, best] + [('' if math.isnan(f1s[v]) else round(f1s[v], 4)) for v in ROUTABLE])
    else:
        target.append([k, 0, 'no_reference'] + ['' for _ in ROUTABLE])
    if (n + 1) % 100 == 0: print(f'  {n+1}/{len(KEYS)}  ({time.time()-t0:.0f}s)')
print('done. images with zero clean-reference objects:', n_noref)

In [ ]:
# write per-image CSV, per-version summary, and the Stage-4 detection target
import csv, numpy as np
with open('/kaggle/working/metrics/detection_per_image.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['relative_path','version','n_objects','mean_conf','n_ref_objects',
                'tp','fp','fn','precision','recall','f1_agreement'])
    w.writerows(rows)

# summary per version (means over images; F1/recall averaged only where a reference exists)
by = {v: [] for v in VERS}
for r in rows: by[r[1]].append(r)
with open('/kaggle/working/metrics/detection_summary.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['version','mean_n_objects','mean_conf','mean_precision','mean_recall',
                'mean_f1_agreement','total_fp','total_fn','n_images'])
    for v in VERS:
        rs = by[v]
        f1v = [float(r[10]) for r in rs if r[10] != '']
        rcv = [float(r[9]) for r in rs if r[9] != '']
        w.writerow([v,
                    round(np.mean([r[2] for r in rs]), 3),
                    round(np.mean([r[3] for r in rs]), 4),
                    round(np.mean([float(r[8]) for r in rs]), 4),
                    round(np.mean(rcv), 4) if rcv else '',
                    round(np.mean(f1v), 4) if f1v else '',
                    int(np.sum([r[6] for r in rs])),
                    int(np.sum([r[7] for r in rs])),
                    len(rs)])

with open('/kaggle/working/metrics/stage4_detection_target.csv', 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['relative_path','n_ref_objects','best_version'] +
               [f'f1__{v}' for v in ['degraded','dit','rcdnet','rcdnet_then_dit','dit_then_rcdnet']])
    w.writerows(target)

# class histogram per version -> 'top false classes' (clean has ~no objects, so these are ~all false)
with open('/kaggle/working/metrics/detection_class_counts.csv', 'w', newline='') as f:
    w = csv.writer(f); w.writerow(['version', 'class', 'count'])
    for v in VERS:
        for cls, ct in cls_count[v].most_common():
            w.writerow([v, cls, ct])
print('\ntop hallucinated classes on degraded:')
for cls, ct in cls_count['degraded'].most_common(8): print(f'   {cls:16s} {ct}')
print('top hallucinated classes on rcdnet:')
for cls, ct in cls_count['rcdnet'].most_common(8): print(f'   {cls:16s} {ct}')

print(open('/kaggle/working/metrics/detection_summary.csv').read())

In [ ]:
import os
os.system('cd /kaggle/working && rm -f metrics.zip && zip -r metrics.zip metrics -q')
print('wrote metrics.zip')
# quick look at the detection target distribution
import csv, collections
t = list(csv.DictReader(open('/kaggle/working/metrics/stage4_detection_target.csv')))
c = collections.Counter(r['best_version'] for r in t)
print('best-version-by-detection distribution:')
for v, n in c.most_common(): print(f'   {v:18s} {n}')
print('rows:', len(t))
print('DOWNLOAD metrics.zip from the Output panel.')